In [2]:
!pip install -U scikit-learn scikeras tensorflow pandas numpy

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 100.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 572.9/572.9 MB 858.6 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 109.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.7/16.7 MB 96.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 130.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 343.2/343.2 kB 33.1 MB/s eta 0:00:00
  Attempting uninstall: protobuf
    Found existing installation: protobuf 5.29.6
    Uninstalling protobuf-5.29.6:
      Successfully uninstalled protobuf-5.29.6
  Attempting uninstall: numpy
    Found existing installation: numpy 2.1.3
    Uninstalling numpy-2.1.3:
      Successfully uninstalled numpy-2.1.3
  Attempting uninstall: pandas
    Found existing installation: pandas 2.2.3
    Uninstalling pandas-2.2.3:
      Successfully unins

In [2]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Load dataset
data = pd.read_csv("/content/diabetes (1).csv")

In [3]:
print("First 5 rows:")
print(data.head())

print("\nDataset Shape:")
print(data.shape)

print("\nDataset Information:")
print(data.info())

print("\nStatistical Summary:")
print(data.describe())

print("\nMissing Values:")
print(data.isnull().sum())

First 5 rows:
   Pregnancies  Glucose  BloodPressure  SkinThickness  Insulin   BMI  \
0            6      148             72             35        0  33.6   
1            1       85             66             29        0  26.6   
2            8      183             64              0        0  23.3   
3            1       89             66             23       94  28.1   
4            0      137             40             35      168  43.1   

   DiabetesPedigreeFunction  Age  Outcome  
0                     0.627   50        1  
1                     0.351   31        0  
2                     0.672   32        1  
3                     0.167   21        0  
4                     2.288   33        1  

Dataset Shape:
(768, 9)

Dataset Information:
<class 'pandas.DataFrame'>
RangeIndex: 768 entries, 0 to 767
Data columns (total 9 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Pregnancies               768 non-

In [4]:
# Separate input features and target

X = data.drop("Outcome", axis=1)
y = data["Outcome"]

print("Features:")
print(X.head())

print("\nTarget:")
print(y.head())

Features:
   Pregnancies  Glucose  BloodPressure  SkinThickness  Insulin   BMI  \
0            6      148             72             35        0  33.6   
1            1       85             66             29        0  26.6   
2            8      183             64              0        0  23.3   
3            1       89             66             23       94  28.1   
4            0      137             40             35      168  43.1   

   DiabetesPedigreeFunction  Age  
0                     0.627   50  
1                     0.351   31  
2                     0.672   32  
3                     0.167   21  
4                     2.288   33  

Target:
0    1
1    0
2    1
3    0
4    1
Name: Outcome, dtype: int64


In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training data:", X_train.shape)
print("Testing data:", X_test.shape)

Training data: (614, 8)
Testing data: (154, 8)


In [6]:
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)

X_test = scaler.transform(X_test)

print("Training data after scaling:")
print(X_train[:5])

Training data after scaling:
[[-0.85135507 -0.98013068 -0.40478372 -0.55397256 -0.33131928 -0.60767846
   0.31079384 -0.79216928]
 [ 0.35657564  0.16144422  0.46536842  0.392787   -0.52639809 -0.30213902
  -0.11643851  0.56103382]
 [-0.5493724  -0.50447447 -0.62232176  1.21331196  0.14244354  0.3725939
  -0.76486207 -0.70759409]
 [-0.85135507  0.7956525  -0.73109078 -1.31138021 -0.73076636 -0.28940821
   0.26231357 -0.36929331]
 [-1.15333775 -0.82157861 -0.29601471  1.15019465  0.24462767  1.60748245
  -0.33762972 -0.96131967]]


In [7]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.optimizers import Adam

from scikeras.wrappers import KerasClassifier

from sklearn.model_selection import GridSearchCV

from sklearn.metrics import accuracy_score, classification_report

In [8]:
def create_model(
    learning_rate=0.001,
    init_mode="uniform",
    activation="relu",
    dropout_rate=0.2,
    neurons=12
):

    model = Sequential()

    # Hidden Layer
    model.add(
        Dense(
            neurons,
            input_dim=8,
            kernel_initializer=init_mode,
            activation=activation
        )
    )

    # Dropout Layer
    model.add(
        Dropout(dropout_rate)
    )

    # Output Layer
    model.add(
        Dense(
            1,
            kernel_initializer=init_mode,
            activation="sigmoid"
        )
    )

    # Adam Optimizer
    optimizer = Adam(
        learning_rate=learning_rate
    )

    # Compile Model
    model.compile(
        optimizer=optimizer,
        loss="binary_crossentropy",
        metrics=["accuracy"]
    )

    return model

In [9]:
model = KerasClassifier(
    model=create_model,
    verbose=0
)

In [10]:
param_grid = {

    # Tune batch size
    "batch_size": [10, 20],

    # Tune training epochs
    "epochs": [50, 100],

    # Tune learning rate
    "model__learning_rate": [0.001, 0.01],

    # Tune weight initialization
    "model__init_mode": ["uniform", "normal"],

    # Tune activation function
    "model__activation": ["relu", "tanh"],

    # Tune dropout regularization
    "model__dropout_rate": [0.0, 0.2],

    # Tune number of neurons
    "model__neurons": [8, 16]
}

In [11]:
grid = GridSearchCV(
    estimator=model,
    param_grid=param_grid,
    cv=3,
    scoring="accuracy",
    n_jobs=-1
)

In [ ]:
import time

print("========================================")
print("Starting Grid Search...")
print("Please wait...")
print("========================================")

start_time = time.time()

grid_result = grid.fit(X_train, y_train)

end_time = time.time()

total_time = end_time - start_time

print("\n========================================")
print("Grid Search Completed!")
print("========================================")

if total_time < 60:
    print(f"Total time taken: {total_time:.2f} seconds")
else:
    minutes = int(total_time // 60)
    seconds = int(total_time % 60)
    print(f"Total time taken: {minutes} minutes {seconds} seconds")

Starting Grid Search...
Please wait...


In [ ]:
print("\nBest Parameters:")
print(grid_result.best_params_)

In [ ]:
print("\nBest Cross-Validation Accuracy:")
print(grid_result.best_score_)

In [ ]:
best_model = grid_result.best_estimator_

y_pred = best_model.predict(X_test)

y_pred = np.array(y_pred).astype(int).reshape(-1)

accuracy = accuracy_score(y_test, y_pred)

print("\nTest Accuracy:")
print(accuracy)

In [ ]:
print("\nClassification Report:")

print(
    classification_report(
        y_test,
        y_pred
    )
)

In [ ]:
results = pd.DataFrame(grid_result.cv_results_)

results = results[
    [
        "mean_test_score",
        "std_test_score",
        "param_batch_size",
        "param_epochs",
        "param_model__learning_rate",
        "param_model__init_mode",
        "param_model__activation",
        "param_model__dropout_rate",
        "param_model__neurons"
    ]
]

results = results.sort_values(
    by="mean_test_score",
    ascending=False
)

print(results.head(10))